# AIC26 · PE-Core-G14-448 encoder server — Google Colab A100

Server encode phía query mà backend retrieval gọi tới (`PE_ENCODER_URL`), public qua custom domain **`https://pe.baoencoder.site`** giống notebook Qwen encoder:

| Route | Model | Dùng cho |
|---|---|---|
| `POST /encode-text` | PE-Core-G14-448 text tower → 1280-d | mọi search hình ảnh bằng PE Core (T-KIS, V-KIS, QA, TRAKE) |
| `POST /encode-image` | PE-Core-G14-448 vision tower → 1280-d | **V-KIS sketch** (canvas vẽ tay) trên cả BTC và InfoShot++ |
| `POST /encode-audio-text` | GLAP (`mispeech/GLAP`) text → 1024-d | audio vector search (tuỳ chọn, `ENABLE_GLAP`) |
| `GET /health` | — | health check của backend/tunnel (public) |

**Cách chạy**
1. *Runtime → Change runtime type →* **A100 GPU** (bật High-RAM nếu có).
2. *Secrets* (🔑 bên trái, bật *Notebook access*) — cùng kiểu với notebook Qwen encoder:
   - `CF_TUNNEL_TOKEN_PE` — token của **tunnel riêng cho PE** → custom domain cố định `https://pe.baoencoder.site` qua mọi runtime (xem mục 9). Không có → quick tunnel `*.trycloudflare.com`, URL đổi mỗi lần.
   - `PE_ENCODER_TOKEN` — bearer token dùng chung với backend (`PE_ENCODER_TOKEN` trong `backend/.env`). Không có → notebook tự sinh token ngẫu nhiên chỉ sống trong runtime này.
   - `PE_ENCODER_HOSTNAME` — (tuỳ chọn) hostname khác `pe.baoencoder.site`.
   - `HF_TOKEN` — (tuỳ chọn) nếu Hugging Face giới hạn tốc độ tải checkpoint 9.7 GB.
3. *Runtime → Run all.* Cell tunnel in ra đúng các dòng cần có trong `backend/.env`; cell cuối canh chừng server + tunnel (bấm Stop để lấy lại notebook, server vẫn chạy).

**Hợp đồng số học — phải khớp index Milvus** (xem `HF_BUCKET/PE_Core_G14_448_AIC2026_Retrieval_Handoff.md`, `Batch2/PE_Core_G14_448_Batch2_AIC2026_Retrieval_Handoff.md`):
weights **FP32** + CUDA autocast **FP16**, eager (**không** `torch.compile`), resize **squash** 448×448 bilinear (không centre crop), L2-normalize ở FP32.
A100 hỗ trợ bf16 nhưng **không đổi sang bf16**: vector query sẽ lệch khỏi không gian đã index.

## 1. Cấu hình

In [ ]:
# ===== Cấu hình =====
import os, re, secrets

PORT = 8000                        # Public Hostname của tunnel phải trỏ về HTTP localhost:8000
PUBLIC_HOSTNAME_DEFAULT = "pe.baoencoder.site"  # custom domain; secret PE_ENCODER_HOSTNAME ghi đè
ENABLE_GLAP = True                 # /encode-audio-text cho audio vector search; False = setup nhanh hơn ~2 phút
PE_MODEL = "PE-Core-G14-448"
PE_REPO_URL = "https://github.com/facebookresearch/perception_models.git"
PE_REPO_DIR = "/content/perception_models"
CLOUDFLARED = "/content/cloudflared"
CLOUDFLARED_LOG = "/content/cloudflared.log"
MAX_TEXTS_PER_REQUEST = 64
MAX_IMAGES_PER_REQUEST = 8         # sketch search gửi 1 ảnh
MAX_IMAGE_BYTES = 8 * 1024 * 1024  # mỗi ảnh sau khi decode base64
# Giữ checkpoint 9.7 GB qua các session: trỏ vào một thư mục Drive (sẽ tự mount).
HF_CACHE_DIR = None                # vd "/content/drive/MyDrive/hf-cache"


def colab_secret(name):
    """Secret của Colab (🔑, Notebook access bật), rồi tới biến môi trường, không có thì None."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value and value.strip():
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, "").strip()
    return value or None


HF_TOKEN = colab_secret("HF_TOKEN")
# Colab secrets dùng chung cả account, nên mỗi notebook chạy tunnel riêng có tên secret riêng
# (Qwen encoder: CF_TUNNEL_TOKEN_ENCODER, reranker: CF_TUNNEL_TOKEN_RERANKER). CỐ Ý không đọc
# CF_TUNNEL_TOKEN: đó là tên cũ của tunnel Qwen encoder — chạy nó ở đây biến Colab này thành
# replica của tunnel Qwen và Cloudflare sẽ chia traffic của Qwen sang đây.
TUNNEL_TOKEN = colab_secret("CF_TUNNEL_TOKEN_PE")
PUBLIC_HOSTNAME = (
    (colab_secret("PE_ENCODER_HOSTNAME") or PUBLIC_HOSTNAME_DEFAULT)
    .lower().removeprefix("https://").removeprefix("http://").rstrip("/")
)
if not re.fullmatch(r"[a-z0-9]([a-z0-9-]*[a-z0-9])?(\.[a-z0-9]([a-z0-9-]*[a-z0-9])?)+", PUBLIC_HOSTNAME):
    raise ValueError(f"hostname không hợp lệ: {PUBLIC_HOSTNAME!r}")
if TUNNEL_TOKEN and len(TUNNEL_TOKEN) < 40:
    raise ValueError("CF_TUNNEL_TOKEN_PE không giống token của Cloudflare tunnel (chuỗi eyJ… dài).")
# Bearer token dùng chung với backend. Không có secret → token ngẫu nhiên cho riêng runtime này.
SERVICE_TOKEN = colab_secret("PE_ENCODER_TOKEN") or colab_secret("AIC26_SERVICE_TOKEN")
SERVICE_TOKEN_GENERATED = SERVICE_TOKEN is None
if SERVICE_TOKEN_GENERATED:
    SERVICE_TOKEN = secrets.token_urlsafe(32)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if HF_CACHE_DIR:
    if HF_CACHE_DIR.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    os.makedirs(HF_CACHE_DIR, exist_ok=True)
    os.environ["HF_HOME"] = HF_CACHE_DIR  # phải đặt TRƯỚC khi import huggingface_hub

print("tunnel :", f"named → https://{PUBLIC_HOSTNAME}" if TUNNEL_TOKEN else "quick trycloudflare (chưa có secret CF_TUNNEL_TOKEN_PE; URL đổi mỗi runtime)")
print("bearer :", "tự sinh cho runtime này (in ở mục 9)" if SERVICE_TOKEN_GENERATED else "secret PE_ENCODER_TOKEN")
print("GLAP   :", ENABLE_GLAP)

## 2. Kiểm tra GPU

In [ ]:
import subprocess

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise SystemExit("Không thấy GPU. Runtime → Change runtime type → A100 GPU, rồi chạy lại.")
print(smi.stdout.strip())
if "A100" not in smi.stdout:
    print("⚠ Không phải A100. PE-G14 (~9.7 GB weights FP32) vẫn chạy trên GPU ≥16 GB, chỉ chậm hơn.")

## 3. Cài đặt

Chỉ cài đúng những gì `core.vision_encoder` import. **Không** `pip install -e perception_models`:
`requirements.txt` của repo pin ~30 gói nặng (`numpy==2.1.2`, `torchdata`, `lm-eval`, `wandb`, `decord`, `opencv`, `pillow==11.0.0`…),
trên Colab sẽ hạ numpy dưới runtime đang chạy và có thể kéo torch khác bản CUDA của Colab. Repo được clone và dùng qua `sys.path`.
Torch/torchvision/numpy/Pillow giữ nguyên bản Colab.

In [ ]:
import subprocess, sys, urllib.request


def pip_install(*packages):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("pip install thất bại:\n" + (result.stderr or result.stdout)[-3000:])


pip_install("ftfy", "einops", "timm", "fastapi", "uvicorn>=0.29", "requests")
if ENABLE_GLAP:
    # glap_model (dasheng) cần transformers < 5; 4.53.3 là bản đã chạy GLAP. Cài TRƯỚC khi bất cứ
    # thứ gì import transformers nên không phải restart runtime. PE không dùng transformers.
    pip_install("glap_model==0.0.13", "transformers==4.53.3")

if not os.path.isdir(PE_REPO_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", PE_REPO_URL, PE_REPO_DIR], check=True)
PE_REPO_COMMIT = subprocess.run(
    ["git", "-C", PE_REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()

if not os.path.exists(CLOUDFLARED) or os.path.getsize(CLOUDFLARED) < 10_000_000:
    partial = CLOUDFLARED + ".partial"
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", partial
    )
    if os.path.getsize(partial) < 10_000_000:
        raise RuntimeError("cloudflared tải về nhỏ bất thường")
    os.chmod(partial, 0o755)
    os.replace(partial, CLOUDFLARED)

print("perception_models @", PE_REPO_COMMIT)
print(subprocess.run([CLOUDFLARED, "--version"], capture_output=True, text=True).stdout.strip())

## 4. Load PE-Core-G14-448 (lần đầu tải checkpoint 9.7 GB từ Hugging Face)

In [ ]:
import sys, time

import numpy as np
import torch

sys.path.insert(0, PE_REPO_DIR)
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transforms

PE_DEVICE = "cuda:0"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

t0 = time.time()
# Weights để FP32 như lúc index; chỉ forward chạy trong autocast FP16.
pe_model = pe.CLIP.from_config(PE_MODEL, pretrained=True).to(PE_DEVICE).eval()
pe_tokenizer = pe_transforms.get_text_tokenizer(pe_model.context_length)
# Mặc định của PE = squash 448×448 bilinear, không centre crop — đúng transform đã index keyframe.
pe_image_transform = pe_transforms.get_image_transform(pe_model.image_size)

assert next(pe_model.parameters()).dtype == torch.float32, "weights phải là FP32 để khớp index"
assert pe_model.context_length == 72 and pe_model.image_size == 448, (pe_model.context_length, pe_model.image_size)
GPU_NAME = torch.cuda.get_device_name(0)
print(f"{PE_MODEL} loaded in {time.time() - t0:.0f}s on {GPU_NAME}")
print(f"torch {torch.__version__} | CUDA {torch.version.cuda} | GPU memory {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 5. Hàm encode + sanity check trực tiếp

In [ ]:
import base64, binascii, io, threading

from PIL import Image, ImageDraw

Image.MAX_IMAGE_PIXELS = 25_000_000  # chặn ảnh "bom giải nén"
pe_lock = threading.Lock()           # một GPU: mỗi lúc một forward


class BadImage(ValueError):
    """Payload không phải ảnh inline hợp lệ → HTTP 400, không phải 500."""


@torch.inference_mode()
def encode_pe_text(texts):
    tokens = pe_tokenizer(list(texts)).to(PE_DEVICE)
    with torch.autocast("cuda", dtype=torch.float16):
        feats = pe_model.encode_text(tokens)
    return torch.nn.functional.normalize(feats.float(), dim=-1).cpu().numpy().astype("float32")


def decode_image(payload):
    """Data URL base64 (hoặc base64 trần) → PIL RGB. Không bao giờ fetch URL http(s)."""
    if payload.startswith("data:"):
        header, _, payload = payload.partition(",")
        if ";base64" not in header:
            raise BadImage("chỉ nhận data URL base64")
    try:
        raw = base64.b64decode(payload, validate=True)
    except (binascii.Error, ValueError) as exc:
        raise BadImage(f"base64 không hợp lệ: {exc}") from exc
    if len(raw) > MAX_IMAGE_BYTES:
        raise BadImage(f"ảnh quá lớn ({len(raw)} bytes)")
    try:
        image = Image.open(io.BytesIO(raw))
        image.load()
    except Exception as exc:
        raise BadImage(f"không decode được ảnh: {exc}") from exc
    # convert("RGB") biến pixel trong suốt thành ĐEN; phẳng hoá lên nền trắng thay vì thế.
    if image.mode in ("RGBA", "LA") or (image.mode == "P" and "transparency" in image.info):
        rgba = image.convert("RGBA")
        flat = Image.new("RGB", rgba.size, (255, 255, 255))
        flat.paste(rgba, mask=rgba.getchannel("A"))
        return flat
    return image.convert("RGB")


@torch.inference_mode()
def encode_pe_image(images):
    batch = torch.stack([pe_image_transform(image) for image in images]).to(PE_DEVICE)
    with torch.autocast("cuda", dtype=torch.float16):
        feats = pe_model.encode_image(batch)
    return torch.nn.functional.normalize(feats.float(), dim=-1).cpu().numpy().astype("float32")


def sketch_data_url():
    """Một sketch 16:9 kiểu canvas V-KIS: trời xanh, đường xám, xe đỏ bên trái."""
    img = Image.new("RGB", (1024, 576), "#ffffff")
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, 1024, 300], fill="#87c3ec")
    draw.rectangle([0, 300, 1024, 576], fill="#6aa84f")
    draw.polygon([(560, 576), (640, 310), (700, 310), (900, 576)], fill="#555b63")
    draw.rectangle([80, 400, 330, 520], fill="#e02020")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


# Warm-up (kernel CUDA lần đầu) trước khi mở tunnel: request đầu qua Cloudflare bị cắt ở 100s (lỗi 524).
with pe_lock:
    _txt = encode_pe_text(["a red car on a road under a blue sky", "a piano keyboard close up"])
    _img = encode_pe_image([decode_image(sketch_data_url())])
print("text  :", _txt.shape, "norm", round(float(np.linalg.norm(_txt[0])), 4))
print("image :", _img.shape, "norm", round(float(np.linalg.norm(_img[0])), 4))
print("cos(sketch, 'red car on a road under a blue sky') =", round(float(_img[0] @ _txt[0]), 4))
print("cos(sketch, 'piano keyboard close up')           =", round(float(_img[0] @ _txt[1]), 4))
assert _txt.shape[1] == _img.shape[1] == 1280

## 6. GLAP text encoder cho audio vector search (tuỳ chọn)

Khớp đúng model đã tạo `.glap.npy` (collection `aic26_audio_glap_v1`): `mispeech/GLAP`, 1024-d, L2-normalized.
Chạy trên **CPU**: query text rất nhẹ, và `glap.encode_text` không tự đưa token ids lên GPU
(lỗi *"Expected all tensors to be on the same device"*). Backend gọi `{PE_ENCODER_URL}/encode-audio-text`
với cùng `PE_ENCODER_TOKEN`, trừ khi đặt `GLAP_ENCODER_URL` riêng.

In [ ]:
glap_model = None
glap_lock = threading.Lock()

if ENABLE_GLAP:
    from glap_model import glap_inference

    glap_model = glap_inference()
    try:
        glap_model = glap_model.to("cpu").eval()
    except Exception:
        pass


@torch.inference_mode()
def encode_glap_text(texts):
    emb = glap_model.encode_text(list(texts)).float()
    emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return emb.detach().cpu().numpy().astype("float32")


if glap_model is not None:
    _g = encode_glap_text(["a car horn honking in traffic", "piano music"])
    print("GLAP text:", _g.shape, "norm", round(float(np.linalg.norm(_g[0])), 4))
else:
    print("GLAP tắt: /encode-audio-text sẽ trả 503.")

## 7. FastAPI server (mọi route khai báo trước khi start)

`/encode-*` bắt buộc `Authorization: Bearer <PE_ENCODER_TOKEN>`; `/health` public để backend và tunnel kiểm tra
không tốn một lượt GPU. Chạy lại cell này sẽ dừng server cũ trước khi mở server mới trên cùng port.

In [ ]:
import hmac, time
from typing import List, Optional

import requests
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="AIC26 PE-Core-G14-448 encoder")
ROUTES = ["/encode-text", "/encode-image"] + (["/encode-audio-text"] if glap_model is not None else [])


def check_auth(authorization: Optional[str]):
    """Bearer bắt buộc trên mọi route encode; /health để public cho health check của backend/tunnel."""
    scheme, _, credential = (authorization or "").partition(" ")
    if scheme.lower() != "bearer" or not hmac.compare_digest(credential, SERVICE_TOKEN):
        raise HTTPException(status_code=401, detail="Invalid bearer token.", headers={"WWW-Authenticate": "Bearer"})


class EncodeTextRequest(BaseModel):
    texts: List[str] = Field(..., min_length=1, max_length=MAX_TEXTS_PER_REQUEST)


class EncodeImageRequest(BaseModel):
    images: List[str] = Field(..., min_length=1, max_length=MAX_IMAGES_PER_REQUEST)


def vectors_response(model, vectors, t0):
    return {
        "model": model,
        "dim": int(vectors.shape[1]),
        "latency_ms": round((time.time() - t0) * 1000, 2),
        "vectors": vectors.tolist(),
    }


@app.get("/health")
def health():
    return {
        "ok": True,
        "model": f"facebook/{PE_MODEL}",
        "device": PE_DEVICE,
        "gpu": GPU_NAME,
        "context_length": pe_model.context_length,
        "image_size": pe_model.image_size,
        "precision": "fp32 weights + autocast fp16",
        "routes": ROUTES,
        "glap": glap_model is not None,
        "perception_models": PE_REPO_COMMIT,
    }


@app.post("/encode-text")
def api_encode_text(req: EncodeTextRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    t0 = time.time()
    with pe_lock:
        vectors = encode_pe_text(req.texts)
    return vectors_response(f"facebook/{PE_MODEL}", vectors, t0)


@app.post("/encode-image")
def api_encode_image(req: EncodeImageRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    t0 = time.time()
    try:
        images = [decode_image(item) for item in req.images]
    except BadImage as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    with pe_lock:
        vectors = encode_pe_image(images)
    return vectors_response(f"facebook/{PE_MODEL}", vectors, t0)


@app.post("/encode-audio-text")
def api_encode_audio_text(req: EncodeTextRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    if glap_model is None:
        raise HTTPException(status_code=503, detail="GLAP is disabled on this server (ENABLE_GLAP=False)")
    t0 = time.time()
    with glap_lock:
        vectors = encode_glap_text(req.texts)
    return vectors_response("mispeech/GLAP", vectors, t0)


def start_server():
    """(Re)start uvicorn in a background thread and wait until /health answers."""
    global pe_server, pe_server_thread
    old = globals().get("pe_server")
    if old is not None:
        old.should_exit = True
        globals()["pe_server_thread"].join(timeout=15)
    pe_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
    pe_server_thread = threading.Thread(target=pe_server.run, daemon=True)
    pe_server_thread.start()
    for _ in range(60):
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).ok:
                return
        except requests.RequestException:
            pass
        time.sleep(0.5)
    raise RuntimeError(f"server không lên trên port {PORT}")


start_server()
print(f"server: http://127.0.0.1:{PORT}  routes: {ROUTES}")

## 8. Tự kiểm tra qua HTTP (local)

In [ ]:
AUTH = {"Authorization": f"Bearer {SERVICE_TOKEN}"}
LOCAL = f"http://127.0.0.1:{PORT}"

print(requests.get(f"{LOCAL}/health", timeout=10).json())

r = requests.post(f"{LOCAL}/encode-text", json={"texts": ["a flooded street with people pushing motorbikes"]}, headers=AUTH, timeout=60)
r.raise_for_status()
text_vec = np.array(r.json()["vectors"][0])
assert len(text_vec) == 1280 and abs(np.linalg.norm(text_vec) - 1) < 1e-3

sketch = sketch_data_url()
r = requests.post(f"{LOCAL}/encode-image", json={"images": [sketch]}, headers=AUTH, timeout=60)
r.raise_for_status()
http_vec = np.array(r.json()["vectors"][0])
with pe_lock:
    direct_vec = encode_pe_image([decode_image(sketch)])[0]
print("encode-image: dim", len(http_vec), "| HTTP vs trực tiếp cos =", round(float(http_vec @ direct_vec), 6))
assert float(http_vec @ direct_vec) > 0.9999

bad = requests.post(f"{LOCAL}/encode-image", json={"images": ["https://example.com/x.png"]}, headers=AUTH, timeout=30)
assert bad.status_code == 400, bad.status_code  # URL bị từ chối, không fetch
assert requests.post(f"{LOCAL}/encode-text", json={"texts": ["x"]}, timeout=30).status_code == 401
assert requests.post(f"{LOCAL}/encode-text", json={"texts": ["x"]}, headers={"Authorization": "Bearer sai"}, timeout=30).status_code == 401

if glap_model is not None:
    r = requests.post(f"{LOCAL}/encode-audio-text", json={"texts": ["piano music"]}, headers=AUTH, timeout=60)
    r.raise_for_status()
    print("encode-audio-text: dim", r.json()["dim"])


def median_ms(route, body, n):
    times = []
    for _ in range(n):
        t0 = time.time()
        requests.post(f"{LOCAL}{route}", json=body, headers=AUTH, timeout=60).raise_for_status()
        times.append((time.time() - t0) * 1000)
    return round(float(np.median(times)), 1)


print("median latency: encode-text", median_ms("/encode-text", {"texts": ["a red car on a road"]}, 20), "ms",
      "| encode-image", median_ms("/encode-image", {"images": [sketch]}, 10), "ms")
print("✅ server OK (auth, validation, parity)")

## 9. Cloudflare named tunnel + custom domain `pe.baoencoder.site`

Giống notebook Qwen encoder: một **named tunnel riêng** giữ hostname cố định, nên `PE_ENCODER_URL` chỉ phải ghi vào backend một lần. Làm một lần trên Cloudflare:

1. *Zero Trust → Networks → Tunnels → Create a tunnel → Cloudflared*. Tunnel này **chỉ** dành cho PE server — không dùng lại token của tunnel Qwen encoder, reranker hay tunnel `video.baoencoder.site`.
2. Tab *Public Hostname → Add*: Subdomain **`pe`**, Domain **`baoencoder.site`**, Type **HTTP**, URL **`localhost:8000`**. Hostname này phải trùng `PUBLIC_HOSTNAME` (mặc định `pe.baoencoder.site`).
3. Copy token (chuỗi `eyJ…` trong lệnh cài connector) vào Colab Secret **`CF_TUNNEL_TOKEN_PE`**, bật Notebook access. **Không** chạy lệnh `cloudflared.exe service install …` của tunnel này trên máy Windows: notebook chính là connector.
4. `backend/.env`: `PE_ENCODER_URL=https://pe.baoencoder.site` và `PE_ENCODER_TOKEN=<giá trị secret PE_ENCODER_TOKEN>` (GLAP dùng chung cả hai).

**Vì sao phải tunnel riêng:** hai `cloudflared` chạy *cùng* một tunnel được đăng ký là replica HA của một origin, và Cloudflare chia request của **mọi** hostname của tunnel đó cho cả hai kết nối. Cell dưới đọc cấu hình route mà Cloudflare đẩy xuống connector: nếu tunnel phục vụ hostname khác mà không có `pe.baoencoder.site`, cell **dừng cloudflared ngay** và báo lỗi thay vì cướp traffic của server kia.

Không có `CF_TUNNEL_TOKEN_PE` → quick tunnel `*.trycloudflare.com` (URL đổi mỗi runtime). Log của cloudflared ghi ra file `/content/cloudflared.log`, không dùng PIPE: pipe không ai đọc sẽ đầy (~64 KB) và làm cloudflared bị treo — tunnel chết âm thầm.

In [ ]:
import json

LOCAL_ORIGIN = re.compile(r"^(?:http://)?(?:localhost|127\.0\.0\.1|0\.0\.0\.0)(?::(\d+))?/?$")
TUNNEL_MODE = "named" if TUNNEL_TOKEN else "quick"


class WrongTunnel(RuntimeError):
    """Token thuộc tunnel của một server khác."""


def tunnel_routes(log_text):
    """Route Cloudflare đẩy xuống connector (tunnel quản lý từ dashboard); None nếu chưa nhận được.

    cloudflared ghi: INF Updated to new configuration config="{\"ingress\":[...]}" version=N
    """
    routes = None
    for match in re.finditer(r'config=("(?:[^"\\]|\\.)*")', log_text):
        try:
            config = json.loads(json.loads(match.group(1)))
        except (ValueError, TypeError):
            continue
        if isinstance(config, dict):
            routes = [r for r in (config.get("ingress") or []) if isinstance(r, dict) and r.get("hostname")]
    return routes


def serves_this_server(route):
    match = LOCAL_ORIGIN.match(str(route.get("service") or ""))
    return bool(match) and int(match.group(1) or 80) == PORT


def check_routes(routes):
    """Đối chiếu route của tunnel với PUBLIC_HOSTNAME. Raise WrongTunnel nếu token là của server khác."""
    if routes is None:
        print(f"⚠ Chưa đọc được cấu hình route từ Cloudflare; kiểm tra tay: {PUBLIC_HOSTNAME} → HTTP localhost:{PORT}")
        return
    mine = [r for r in routes if r["hostname"].lower() == PUBLIC_HOSTNAME]
    others = [f"{r['hostname']} → {r.get('service')}" for r in routes if r["hostname"].lower() != PUBLIC_HOSTNAME]
    if others and not mine:
        raise WrongTunnel(
            f"CF_TUNNEL_TOKEN_PE là token của tunnel đang phục vụ: {'; '.join(others)} — không có {PUBLIC_HOSTNAME}. "
            "Chạy nó ở đây sẽ biến Colab thành replica và Cloudflare chia traffic của server kia sang đây. "
            "Tạo một tunnel riêng cho PE (mục 9) rồi đặt token của nó vào CF_TUNNEL_TOKEN_PE."
        )
    if not mine:
        print("⚠ Tunnel chưa có Public Hostname nào. Cloudflare → Zero Trust → Networks → Tunnels → tunnel này →")
        print(f"  Public Hostname → Add: {PUBLIC_HOSTNAME} · Type HTTP · URL localhost:{PORT}")
    for route in mine:
        if not serves_this_server(route):
            print(f"⚠ {route['hostname']} đang trỏ tới {route.get('service')}; phải là HTTP localhost:{PORT}")
    if others:
        print(f"⚠ Tunnel này còn phục vụ: {'; '.join(others)}. Nếu máy khác cũng chạy connector của tunnel này,")
        print("  Cloudflare chia request giữa hai máy → lỗi ngẫu nhiên. Nên để tunnel chỉ phục vụ PE.")


def stop_tunnel():
    old = globals().get("tunnel_proc")
    if old is not None and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=10)
        except subprocess.TimeoutExpired:
            old.kill()


def start_tunnel():
    """Start cloudflared; trả về URL public (named: https://PUBLIC_HOSTNAME, quick: URL trycloudflare)."""
    global tunnel_proc
    stop_tunnel()  # chỉ dừng tunnel do notebook này mở, không đụng cloudflared khác
    env = dict(os.environ)
    command = [CLOUDFLARED, "tunnel", "--no-autoupdate", "--protocol", "http2", "--loglevel", "info"]
    if TUNNEL_MODE == "named":
        env["TUNNEL_TOKEN"] = TUNNEL_TOKEN  # qua env, không lộ token trong danh sách tiến trình
        command.append("run")  # route (ingress) nằm trên dashboard, nên không có --url
    else:
        command += ["--url", f"http://localhost:{PORT}"]
    with open(CLOUDFLARED_LOG, "w") as log:
        tunnel_proc = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env)

    deadline = time.time() + 90
    registered_at = None
    while time.time() < deadline:
        text = open(CLOUDFLARED_LOG, errors="replace").read()
        if tunnel_proc.poll() is not None:
            raise RuntimeError("cloudflared đã thoát:\n" + text[-2000:])
        if TUNNEL_MODE == "named" and "Registered tunnel connection" in text:
            registered_at = registered_at or time.time()
            # Cấu hình route tới ngay sau khi nối; chờ thêm tối đa 20s.
            if "Updated to new configuration" in text or time.time() - registered_at > 20:
                try:
                    check_routes(tunnel_routes(text))
                except WrongTunnel:
                    stop_tunnel()
                    raise
                return f"https://{PUBLIC_HOSTNAME}"
        if TUNNEL_MODE == "quick":
            # api.trycloudflare.com xuất hiện trong thông báo lỗi, không phải URL của tunnel.
            urls = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", text) if not u.startswith("https://api.")]
            if urls:
                return urls[0]
        time.sleep(0.5)
    stop_tunnel()
    raise TimeoutError("cloudflared chưa kết nối sau 90s:\n" + open(CLOUDFLARED_LOG, errors="replace").read()[-2000:])


def public_ok(url, attempts=1):
    for attempt in range(attempts):
        try:
            if requests.get(f"{url}/health", timeout=15).ok:
                return True
        except requests.RequestException:
            pass
        if attempt + 1 < attempts:
            time.sleep(3)
    return False


def public_encode_ok(url):
    try:
        r = requests.post(f"{url}/encode-text", json={"texts": ["a red car on a road"]}, headers=AUTH, timeout=60)
        return r.ok and r.json().get("dim") == 1280
    except (requests.RequestException, ValueError):
        return False


def print_backend_env(url):
    print("\n===== backend/.env (rồi Settings → Reload, hoặc POST /api/config/reload) =====")
    print(f"PE_ENCODER_URL={url}")
    if SERVICE_TOKEN_GENERATED:
        print(f"PE_ENCODER_TOKEN={SERVICE_TOKEN}")
        print("# ⚠ token tự sinh, đổi mỗi runtime — đặt secret PE_ENCODER_TOKEN để giữ cố định")
    else:
        print("PE_ENCODER_TOKEN=<giá trị secret PE_ENCODER_TOKEN>")
    print("# GLAP (/encode-audio-text) dùng chung URL + token này trừ khi đặt GLAP_ENCODER_URL")
    print("=" * 80)


PUBLIC_URL = start_tunnel()
# DNS/route có thể mất vài chục giây mới trả lời qua internet.
if public_ok(PUBLIC_URL, attempts=20):
    print("remote /health: OK", "| remote /encode-text (bearer):", "OK" if public_encode_ok(PUBLIC_URL) else "LỖI")
elif TUNNEL_MODE == "named":
    print(f"⚠ Chưa vào được {PUBLIC_URL}/health: kiểm tra Public Hostname {PUBLIC_HOSTNAME} → HTTP localhost:{PORT}.")
    print("  Cell canh chừng sẽ báo khi URL trả lời.")
else:
    print("⚠ quick tunnel chưa trả lời; thử lại sau ít phút.")
print("PUBLIC_URL =", PUBLIC_URL)
print_backend_env(PUBLIC_URL)

## 10. Canh chừng server + tunnel (để chạy suốt buổi thi)

Mỗi phút kiểm tra `/health` local, tiến trình cloudflared và URL public:
- server local chết → khởi động lại;
- cloudflared thoát → mở lại tunnel (named tunnel giữ nguyên `https://pe.baoencoder.site`);
- quick tunnel không trả lời 3 lần liên tiếp → mở tunnel mới, **URL mới** được in to rõ để cập nhật `backend/.env`;
- named tunnel đổi trạng thái (vd vừa thêm Public Hostname trên dashboard) → in ra một lần.

Bấm **Stop** để lấy lại notebook — server và tunnel vẫn chạy. Giữ tab Colab mở: Colab có thể ngắt runtime khi tab đóng hoặc idle lâu. Sau khi runtime bị ngắt, chạy lại từ đầu (phải load lại model); với named tunnel URL của backend vẫn giữ nguyên.

In [ ]:
CHECK_EVERY_S = 60
REPORT_EVERY_S = 600
public_failures = 0
public_up = None
last_report = 0.0

print("Đang canh chừng… (Stop để thoát; server + tunnel vẫn chạy)")
try:
    while True:
        time.sleep(CHECK_EVERY_S)
        now = time.strftime("%H:%M:%S")
        try:
            local_ok = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10).ok
        except requests.RequestException:
            local_ok = False
        if not local_ok:
            print(f"{now} ⚠ server local không trả lời → khởi động lại")
            start_server()

        if tunnel_proc.poll() is not None:
            print(f"{now} ⚠ cloudflared đã thoát → mở lại tunnel")
            old_url = PUBLIC_URL
            PUBLIC_URL = start_tunnel()
            public_failures = 0
            if PUBLIC_URL != old_url:
                print("\n" + "!" * 80 + f"\n{now} URL TUNNEL MỚI: {PUBLIC_URL}\n" + "!" * 80)
                print_backend_env(PUBLIC_URL)
        else:
            up = public_ok(PUBLIC_URL)
            if TUNNEL_MODE == "named":
                # URL cố định: mở lại không giúp gì khi route thiếu → chỉ báo khi đổi trạng thái.
                if up != public_up:
                    print(f"{now} {'✅' if up else '⚠'} {PUBLIC_URL} {'trả lời' if up else 'không trả lời (kiểm tra Public Hostname / DNS)'}")
                public_up = up
            else:
                public_failures = 0 if up else public_failures + 1
                if public_failures >= 3:
                    PUBLIC_URL = start_tunnel()
                    public_failures = 0
                    print("\n" + "!" * 80 + f"\n{now} URL TUNNEL MỚI: {PUBLIC_URL}\n" + "!" * 80)
                    print_backend_env(PUBLIC_URL)

        if time.time() - last_report >= REPORT_EVERY_S:
            last_report = time.time()
            print(f"{now} ok · local={'up' if local_ok else 'restarted'} · tunnel={'up' if tunnel_proc.poll() is None else 'down'} · {PUBLIC_URL}")
except KeyboardInterrupt:
    print("Dừng canh chừng. Server và tunnel vẫn chạy.")